In [6]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import optax
import numpy as np
from pyscf import gto, scf, fci
import flax.linen as nn

# 禁用 JIT 调试（正式运行时可注释掉）
# jax.config.update('jax_disable_jit', True)

# ==============================================================================
# 1. H₂ 分子与 FCI 基准
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("=" * 60)
print("FCI 基准能量 (Ha)")
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f}  |  激发能 = {(e - E_fcis[0]) * 27.2114:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# ==============================================================================
# 2. 扩展希尔伯特空间与采样器
# ==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)
hi_ext = hi ** K

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)
sampler = nk.sampler.MetropolisSampler(hi_ext, rule=single_rule, n_chains=16, sweep_size=32)

# ==============================================================================
# 3. Linen 模型定义
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 16

    @nn.compact
    def __call__(self, x):
        # x: (n_spin,)
        x = x.astype(jnp.complex64)
        h = nn.tanh(nn.Dense(self.hidden_dim, dtype=jnp.complex64)(x))
        h = nn.tanh(nn.Dense(self.hidden_dim, dtype=jnp.complex64)(h))
        out = nn.Dense(1, dtype=jnp.complex64)(h)
        return out.squeeze()

class NESModel(nn.Module):
    n_spin_orbitals: int
    K: int
    hidden_dim: int = 16

    def setup(self):
        # 创建 K 个独立的单粒子 ansatz
        self.singles = [SingleStateAnsatz(self.hidden_dim) for _ in range(self.K)]

    def __call__(self, x_batch):
        # x_batch: (batch, K * n_spin) → 返回 log ψ
        batch_size = x_batch.shape[0]
        x_reshaped = x_batch.reshape(batch_size, self.K, self.n_spin_orbitals)

        def log_psi_per_sample(x_sample):
            M = jnp.array([
                [self.singles[j](x_sample[i]) for j in range(self.K)]
                for i in range(self.K)
            ])
            det = jnp.linalg.det(M)
            return jnp.log(det + 1e-10)

        return jax.vmap(log_psi_per_sample)(x_reshaped)

    def eval_single_psi(self, params, j, x):
        """评估第 j 个单粒子波函数 ψ_j(x)"""
        # 这里需要手动提取第 j 个 submodule 的参数并 apply
        # 简便方法：直接调用 self.singles[j].apply(params['params']['singles'][str(j)], x)
        sub_params = {'params': params['params']['singles'][str(j)]}
        return self.singles[j].apply(sub_params, x)

# ==============================================================================
# 4. 局域能量计算（迹）
# ==============================================================================
def Ham_psi(ham, params_j, single_module, x):
    """计算 (H ψ_j)(x)"""
    x_primes, mels = ham.get_conn_padded(x)
    psi_vals = jax.vmap(lambda xp: single_module.apply(params_j, xp))(x_primes)
    return jnp.sum(mels * psi_vals)

def compute_local_energy_single(ham, model, params, x_single):
    """x_single: (K * n_spin,) → 返回迹标量"""
    x_batch = x_single.reshape(model.K, model.n_spin_orbitals)

    # 构建 M 矩阵
    M = jnp.array([
        [model.eval_single_psi(params, j, x_batch[i]) for j in range(model.K)]
        for i in range(model.K)
    ])

    # 构建 H_M 矩阵
    H_mat = []
    for i in range(model.K):
        row = []
        for j in range(model.K):
            sub_params = {'params': params['params']['singles'][str(j)]}
            single_mod = model.singles[j]
            row.append(Ham_psi(ham, sub_params, single_mod, x_batch[i]))
        H_mat.append(row)
    Hp = jnp.array(H_mat, dtype=jnp.complex64)

    M_reg = M + 1e-8 * jnp.eye(model.K, dtype=M.dtype)
    E_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(E_mat).real

batch_local_energy = jax.vmap(compute_local_energy_single, in_axes=(None, None, None, 0))

# ==============================================================================
# 5. 自定义 MCState（重写 expect_and_grad）
# ==============================================================================
class NESMCState(nk.vqs.MCState):
    def expect_and_grad(self, O, *, mutable=None, use_covariance=None):
        print(">>> NESMCState.expect_and_grad called")  # 调试用
        samples = self.samples
        model = self.model
        params = self.parameters

        # 计算所有样本的局域能量
        local_energies = batch_local_energy(ha, model, params, samples)
        loss = local_energies.mean()
        print(f"   Loss (trace avg): {loss:.6f}")

        # 定义损失函数用于梯度计算
        def loss_fn(p):
            les = batch_local_energy(ha, model, p, samples)
            return les.mean()

        grad = jax.grad(loss_fn)(params)
        loss_stats = nk.stats.Stats(mean=loss, variance=0.0, error_of_mean=0.0)
        return loss_stats, grad



FCI 基准能量 (Ha)
E0 = -1.01546825  |  激发能 = 0.0000 eV
E1 = -0.87542794  |  激发能 = 3.8107 eV
E2 = -0.42938376  |  激发能 = 15.9482 eV
E3 = -0.26922131  |  激发能 = 20.3064 eV


In [13]:
class ExtendedHamiltonian(nk.operator.DiscreteOperator):
    def __init__(self, base_hamiltonian, K):
        self.base_ham = base_hamiltonian
        self.K = K
        self._hilbert = nk.hilbert.TensorHilbert(*[base_hamiltonian.hilbert] * K)
        super().__init__(self._hilbert)

    @property
    def dtype(self):
        """数据类型，返回与基哈密顿量一致的类型"""
        return self.base_ham.dtype

    @property
    def max_conn_size(self) -> int:
        # 每个副本独立的最大连接数之和
        return self.base_ham.max_conn_size * self.K

    def get_conn_padded(self, x):
        # 占位实现，返回空连接数组（实际不会被调用）
        n_conn = self.max_conn_size
        xp = jnp.zeros((n_conn, self.hilbert.size), dtype=x.dtype)
        mels = jnp.zeros((n_conn,), dtype=self.dtype)
        return xp, mels

    def get_conn(self, x):
        return self.get_conn_padded(x)

ha_ext = ExtendedHamiltonian(ha, K)

In [14]:
# ==============================================================================
# 6. 初始化模型与变分态
# ==============================================================================
model = NESModel(n_spin_orbitals=4, K=K, hidden_dim=16)

# 生成一个样本输入用于初始化参数
dummy_input = jnp.zeros((1, K * 4), dtype=jnp.int8)
params = model.init(jax.random.PRNGKey(42), dummy_input)

vstate = NESMCState(
    sampler,
    model,
    n_samples=1024,
    n_discard_per_chain=16,
    seed=123,
    variables=params  # 传入初始参数
)

# ==============================================================================
# 7. 优化器与 VMC 驱动
# ==============================================================================
optimizer = nk.optimizer.Sgd(1e-2)

gs = nk.driver.VMC(
    ha_ext,
    optimizer,
    variational_state=vstate
)

print("\n开始训练...")
gs.run(n_iter=100)  # 可先运行少量步数测试




开始训练...


  0%|          | 0/100 [00:00<?, ?it/s]

>>> NESMCState.expect_and_grad called


  0%|          | 0/100 [00:01<?, ?it/s]


TypeError: cannot reshape array of shape (64, 8) (size 512) into shape (2, 4) (size 8)

In [15]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import optax
import numpy as np
from pyscf import gto, scf, fci
import flax.linen as nn

# ==============================================================================
# 1. H₂ 分子与 FCI 基准
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("=" * 60)
print("FCI 基准能量 (Ha)")
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f}  |  激发能 = {(e - E_fcis[0]) * 27.2114:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# ==============================================================================
# 2. 扩展希尔伯特空间与采样器
# ==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)
hi_ext = nk.hilbert.TensorHilbert(*[hi] * K)

print(f"原始 Hilbert 空间大小: {hi.size}")
print(f"扩展 Hilbert 空间大小: {hi_ext.size}")

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

sampler = nk.sampler.MetropolisSampler(
    hi_ext,
    rule=tensor_rule,
    n_chains=16,
    sweep_size=32,
)

# ==============================================================================
# 3. 扩展哈密顿量（占位符）
# ==============================================================================
class ExtendedHamiltonian(nk.operator.DiscreteOperator):
    def __init__(self, base_hamiltonian, K):
        self.base_ham = base_hamiltonian
        self.K = K
        self._hilbert = nk.hilbert.TensorHilbert(*[base_hamiltonian.hilbert] * K)
        super().__init__(self._hilbert)

    @property
    def dtype(self):
        return self.base_ham.dtype

    @property
    def max_conn_size(self) -> int:
        return self.base_ham.max_conn_size * self.K

    def get_conn_padded(self, x):
        n_conn = self.max_conn_size
        xp = jnp.zeros((n_conn, self.hilbert.size), dtype=x.dtype)
        mels = jnp.zeros((n_conn,), dtype=self.dtype)
        return xp, mels

    def get_conn(self, x):
        return self.get_conn_padded(x)

ha_ext = ExtendedHamiltonian(ha, K)

# ==============================================================================
# 4. Linen 模型
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 16

    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex64)
        h = nn.tanh(nn.Dense(self.hidden_dim, dtype=jnp.complex64)(x))
        h = nn.tanh(nn.Dense(self.hidden_dim, dtype=jnp.complex64)(h))
        out = nn.Dense(1, dtype=jnp.complex64)(h)
        return out.squeeze()

class NESModel(nn.Module):
    n_spin_orbitals: int
    K: int
    hidden_dim: int = 16

    def setup(self):
        self.singles = [SingleStateAnsatz(self.hidden_dim) for _ in range(self.K)]

    def __call__(self, x_batch):
        batch_size = x_batch.shape[0]
        x_reshaped = x_batch.reshape(batch_size, self.K, self.n_spin_orbitals)

        def log_psi_per_sample(x_sample):
            M = jnp.array([
                [self.singles[j](x_sample[i]) for j in range(self.K)]
                for i in range(self.K)
            ])
            det = jnp.linalg.det(M)
            return jnp.log(det + 1e-10)

        return jax.vmap(log_psi_per_sample)(x_reshaped)

    def eval_single_psi(self, params, j, x):
        sub_params = {'params': params['params']['singles'][str(j)]}
        return self.singles[j].apply(sub_params, x)

# ==============================================================================
# 5. 局域能量计算
# ==============================================================================
def Ham_psi(ham, params_j, single_module, x):
    x_primes, mels = ham.get_conn_padded(x)
    psi_vals = jax.vmap(lambda xp: single_module.apply(params_j, xp))(x_primes)
    return jnp.sum(mels * psi_vals)

def compute_local_energy_single(ham, model, params, x_single):
    n_spin = model.n_spin_orbitals
    x_batch = x_single.reshape(model.K, n_spin)

    M = jnp.array([
        [model.eval_single_psi(params, j, x_batch[i]) for j in range(model.K)]
        for i in range(model.K)
    ])

    H_mat = []
    for i in range(model.K):
        row = []
        for j in range(model.K):
            sub_params = {'params': params['params']['singles'][str(j)]}
            single_mod = model.singles[j]
            row.append(Ham_psi(ham, sub_params, single_mod, x_batch[i]))
        H_mat.append(row)
    Hp = jnp.array(H_mat, dtype=jnp.complex64)

    M_reg = M + 1e-8 * jnp.eye(model.K, dtype=M.dtype)
    E_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(E_mat).real

batch_local_energy = jax.vmap(compute_local_energy_single, in_axes=(None, None, None, 0))

# ==============================================================================
# 6. 自定义 MCState
# ==============================================================================
class NESMCState(nk.vqs.MCState):
    def expect_and_grad(self, O, *, mutable=None, use_covariance=None):
        print(">>> NESMCState.expect_and_grad called")
        samples = self.samples  # (n_chains, chain_length, hi_ext.size)
        samples_flat = samples.reshape(-1, samples.shape[-1])
        model = self.model
        params = self.parameters

        local_energies = batch_local_energy(ha, model, params, samples_flat)
        loss = local_energies.mean()
        print(f"   Loss (trace avg): {loss:.6f}")

        def loss_fn(p):
            les = batch_local_energy(ha, model, p, samples_flat)
            return les.mean()

        grad = jax.grad(loss_fn)(params)
        loss_stats = nk.stats.Stats(mean=loss, variance=0.0, error_of_mean=0.0)
        return loss_stats, grad



FCI 基准能量 (Ha)
E0 = -1.01546825  |  激发能 = 0.0000 eV
E1 = -0.87542794  |  激发能 = 3.8107 eV
E2 = -0.42938376  |  激发能 = 15.9482 eV
E3 = -0.26922131  |  激发能 = 20.3064 eV
原始 Hilbert 空间大小: 4
扩展 Hilbert 空间大小: 8


In [17]:
# ==============================================================================
# 7. 初始化
# ==============================================================================
model = NESModel(n_spin_orbitals=4, K=K, hidden_dim=16)
dummy_input = jnp.zeros((1, hi_ext.size), dtype=jnp.int8)
params = model.init(jax.random.PRNGKey(42), dummy_input)

vstate = NESMCState(
    sampler,
    model,
    n_samples=1024,
    n_discard_per_chain=16,
    seed=123,
    variables=params
)

optimizer = nk.optimizer.Sgd(1e-2)

gs = nk.driver.VMC(
    ha_ext,
    optimizer,
    variational_state=vstate
)

print("\n开始训练...")
gs.run(n_iter=10)  # 测试

# ==============================================================================
# 8. 提取激发态能量
# ==============================================================================
vstate.n_samples = 4096
vstate.n_discard_per_chain = 32
samples_final = vstate.sample()
samples_flat = samples_final.reshape(-1, hi_ext.size)

def compute_energy_matrix(ham, model, params, x_single):
    n_spin = model.n_spin_orbitals
    x_batch = x_single.reshape(model.K, n_spin)
    M = jnp.array([
        [model.eval_single_psi(params, j, x_batch[i]) for j in range(model.K)]
        for i in range(model.K)
    ])
    H_mat = []
    for i in range(model.K):
        row = []
        for j in range(model.K):
            sub_params = {'params': params['params']['singles'][str(j)]}
            single_mod = model.singles[j]
            row.append(Ham_psi(ham, sub_params, single_mod, x_batch[i]))
        H_mat.append(row)
    Hp = jnp.array(H_mat, dtype=jnp.complex64)
    M_reg = M + 1e-8 * jnp.eye(model.K, dtype=M.dtype)
    return jnp.linalg.solve(M_reg, Hp)

batch_emat = jax.vmap(compute_energy_matrix, in_axes=(None, None, None, 0))
E_mats = batch_emat(ha, model, vstate.parameters, samples_flat)
E_mat_avg = jnp.mean(E_mats, axis=0)

eigvals = jnp.linalg.eigvalsh(E_mat_avg).real
print("\n" + "=" * 60)
print("NES-VMC 计算结果")
for i, e in enumerate(eigvals):
    print(f"E{i} = {e:.8f} Ha  |  与 FCI 偏差: {e - E_fcis[i]:.6f} Ha")


开始训练...


  0%|          | 0/10 [00:00<?, ?it/s]

>>> NESMCState.expect_and_grad called


  0%|          | 0/10 [00:01<?, ?it/s]


KeyError: 'params'